# Portada de Entrega
Universidad del Valle de Guatemala
Inteligencia Artificial - Laboratorio 02
Javier España
Ángel Esquit
Roberto Barreda

Repositorio: https://github.com/Javier-Espana/Lab02-IA.git

# Ejercicio 2: Clasificador Bayesiano Óptimo (Distribuciones Continuas)

## Enunciado

Construir un clasificador bayesiano óptimo para dos distribuciones continuas con densidad exponencial:

- $f_0(x) = f(x|Y=0) = e^{-x}$ para $x \geq 0$ (Exponencial con $\lambda_0 = 1$)
- $f_1(x) = f(x|Y=1) = 3e^{-3x}$ para $x \geq 0$ (Exponencial con $\lambda_1 = 3$)

**Objetivos:**
1. Construir la regla de clasificación
2. Determinar las regiones de clasificación
3. Calcular el error del clasificador

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate
from scipy.stats import expon
from scipy.optimize import fsolve

## 1. Definición de las distribuciones

Recordemos que una distribución exponencial con parámetro $\lambda$ tiene densidad:
$$f(x) = \lambda e^{-\lambda x}, \quad x \geq 0$$

Para este ejercicio:
- **Clase Y=0:** $f_0(x) = e^{-x}$ (Exp con $\lambda_0 = 1$)
- **Clase Y=1:** $f_1(x) = 3e^{-3x}$ (Exp con $\lambda_1 = 3$)

In [ ]:
# Parámetros de las distribuciones
lambda_0 = 1  # Parámetro de la exponencial para Y=0
lambda_1 = 3  # Parámetro de la exponencial para Y=1

# Densidades condicionales
def f0(x):
    """Densidad f(x|Y=0) = exp(-x)"""
    return np.where(x >= 0, lambda_0 * np.exp(-lambda_0 * x), 0)

def f1(x):
    """Densidad f(x|Y=1) = 3*exp(-3x)"""
    return np.where(x >= 0, lambda_1 * np.exp(-lambda_1 * x), 0)

# Probabilidades a priori (asumimos igual probabilidad)
P_Y0 = 0.5  # P(Y=0)
P_Y1 = 0.5  # P(Y=1)

print(f"Distribución para Y=0: Exp({lambda_0}), f₀(x) = {lambda_0}·e^(-{lambda_0}x)")
print(f"Distribución para Y=1: Exp({lambda_1}), f₁(x) = {lambda_1}·e^(-{lambda_1}x)")
print(f"\nProbabilidades a priori:")
print(f"P(Y=0) = {P_Y0}")
print(f"P(Y=1) = {P_Y1}")

In [ ]:
# Visualización de las densidades
x = np.linspace(0, 5, 500)

plt.figure(figsize=(10, 6))
plt.plot(x, f0(x), 'b-', linewidth=2, label=r'$f_0(x) = e^{-x}$ (Y=0)')
plt.plot(x, f1(x), 'r-', linewidth=2, label=r'$f_1(x) = 3e^{-3x}$ (Y=1)')
plt.xlabel('x', fontsize=12)
plt.ylabel('Densidad f(x|Y)', fontsize=12)
plt.title('Densidades Condicionales de las Clases', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xlim(0, 5)
plt.ylim(0, 3.2)
plt.show()

## 2. Regla de Clasificación Bayesiana Óptima

El clasificador bayesiano óptimo clasifica según:

$$\hat{Y}(x) = \begin{cases} 1 & \text{si } f_1(x) \cdot P(Y=1) > f_0(x) \cdot P(Y=0) \\ 0 & \text{en otro caso} \end{cases}$$

Con probabilidades a priori iguales ($P(Y=0) = P(Y=1) = 0.5$), esto se simplifica a:
$$\hat{Y}(x) = \begin{cases} 1 & \text{si } f_1(x) > f_0(x) \\ 0 & \text{en otro caso} \end{cases}$$

### Encontrar el punto de corte (umbral)

Necesitamos encontrar $x^*$ donde $f_0(x^*) \cdot P(Y=0) = f_1(x^*) \cdot P(Y=1)$:

$$e^{-x} \cdot 0.5 = 3e^{-3x} \cdot 0.5$$

Simplificando:
$$e^{-x} = 3e^{-3x}$$
$$e^{2x} = 3$$
$$2x = \ln(3)$$
$$x^* = \frac{\ln(3)}{2}$$

In [ ]:
# Cálculo analítico del umbral
x_star = np.log(lambda_1 / lambda_0) / (lambda_1 - lambda_0)
print(f"Umbral de decisión (analítico):")
print(f"x* = ln({lambda_1}/{lambda_0}) / ({lambda_1} - {lambda_0})")
print(f"x* = ln(3) / 2")
print(f"x* = {x_star:.6f}")
print(f"x* ≈ {x_star:.4f}")

In [ ]:
# Verificación numérica
def diferencia(x):
    return f1(x) * P_Y1 - f0(x) * P_Y0

x_star_numerico = fsolve(diferencia, 0.5)[0]
print(f"\nVerificación numérica: x* = {x_star_numerico:.6f}")

# Verificar que las densidades ponderadas son iguales en x*
print(f"\nVerificación en x* = {x_star:.4f}:")
print(f"f₀(x*) · P(Y=0) = {f0(x_star) * P_Y0:.6f}")
print(f"f₁(x*) · P(Y=1) = {f1(x_star) * P_Y1:.6f}")

## 3. Regiones de Clasificación

Analicemos qué clase es más probable en cada región:

In [ ]:
# Comparar las densidades en diferentes puntos
print("Análisis de las regiones:")
print("="*60)

test_points = [0, 0.3, x_star, 0.7, 1.0, 2.0]
for xp in test_points:
    f0_val = f0(xp) * P_Y0
    f1_val = f1(xp) * P_Y1
    clasificacion = 1 if f1_val > f0_val else 0
    print(f"x = {xp:.4f}: f₀(x)·P(Y=0) = {f0_val:.4f}, f₁(x)·P(Y=1) = {f1_val:.4f} → Clase {clasificacion}")

In [ ]:
# Conclusión sobre las regiones
print("\n" + "="*60)
print("REGIONES DE CLASIFICACIÓN")
print("="*60)
print(f"\nPara x < x* = {x_star:.4f}:")
print(f"  f₁(x)·P(Y=1) > f₀(x)·P(Y=0)")
print(f"  → Clasificar como Y = 1")
print(f"\nPara x > x* = {x_star:.4f}:")
print(f"  f₀(x)·P(Y=0) > f₁(x)·P(Y=1)")
print(f"  → Clasificar como Y = 0")

print(f"\n" + "="*60)
print("REGLA DE CLASIFICACIÓN:")
print("="*60)
print(f"\n       ⎧ 1   si x < {x_star:.4f} (≈ ln(3)/2)")
print(f"h(x) = ⎨")
print(f"       ⎩ 0   si x ≥ {x_star:.4f}")

In [ ]:
# Visualización de las regiones de clasificación
x = np.linspace(0, 4, 1000)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico 1: Densidades ponderadas
ax1 = axes[0]
ax1.plot(x, f0(x) * P_Y0, 'b-', linewidth=2, label=r'$f_0(x) \cdot P(Y=0)$')
ax1.plot(x, f1(x) * P_Y1, 'r-', linewidth=2, label=r'$f_1(x) \cdot P(Y=1)$')
ax1.axvline(x=x_star, color='green', linestyle='--', linewidth=2, label=f'x* = {x_star:.4f}')
ax1.fill_between(x[x < x_star], 0, f1(x[x < x_star]) * P_Y1, alpha=0.3, color='red', label='Región Y=1')
ax1.fill_between(x[x >= x_star], 0, f0(x[x >= x_star]) * P_Y0, alpha=0.3, color='blue', label='Región Y=0')
ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('Densidad ponderada', fontsize=12)
ax1.set_title('Densidades Ponderadas y Regiones de Clasificación', fontsize=14)
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 4)
ax1.set_ylim(0, 1.6)

# Gráfico 2: Regiones de decisión
ax2 = axes[1]
ax2.axvspan(0, x_star, alpha=0.4, color='red', label='Región R₁: Clasificar Y=1')
ax2.axvspan(x_star, 4, alpha=0.4, color='blue', label='Región R₀: Clasificar Y=0')
ax2.axvline(x=x_star, color='green', linestyle='--', linewidth=3, label=f'Frontera: x* = {x_star:.4f}')
ax2.set_xlabel('x', fontsize=12)
ax2.set_title('Regiones de Decisión', fontsize=14)
ax2.legend(loc='upper right', fontsize=11)
ax2.set_xlim(0, 4)
ax2.set_ylim(0, 1)
ax2.set_yticks([])

# Añadir texto con las regiones
ax2.text(x_star/2, 0.5, 'Y = 1', fontsize=20, ha='center', va='center', fontweight='bold')
ax2.text((x_star + 4)/2, 0.5, 'Y = 0', fontsize=20, ha='center', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('ejercicio2_regiones_clasificacion.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Cálculo del Error del Clasificador

El error del clasificador bayesiano se compone de dos tipos de errores:

1. **Error Tipo I (Falso Negativo):** Clasificar como Y=0 cuando realmente es Y=1
   $$P(\hat{Y}=0 | Y=1) \cdot P(Y=1) = P(X > x^* | Y=1) \cdot P(Y=1)$$

2. **Error Tipo II (Falso Positivo):** Clasificar como Y=1 cuando realmente es Y=0
   $$P(\hat{Y}=1 | Y=0) \cdot P(Y=0) = P(X < x^* | Y=0) \cdot P(Y=0)$$

El error total es:
$$\text{Error} = P(X > x^* | Y=1) \cdot P(Y=1) + P(X < x^* | Y=0) \cdot P(Y=0)$$

In [ ]:
# Cálculo analítico del error

# Para una distribución exponencial Exp(λ):
# P(X < x) = 1 - e^(-λx) (CDF)
# P(X > x) = e^(-λx)

# Error 1: P(X > x* | Y=1) · P(Y=1)
# Para Y=1: Exp(3), P(X > x*) = e^(-3x*)
P_error_1 = np.exp(-lambda_1 * x_star) * P_Y1

# Error 2: P(X < x* | Y=0) · P(Y=0)  
# Para Y=0: Exp(1), P(X < x*) = 1 - e^(-x*)
P_error_2 = (1 - np.exp(-lambda_0 * x_star)) * P_Y0

print("CÁLCULO DEL ERROR DEL CLASIFICADOR")
print("="*60)
print(f"\nUmbral de decisión: x* = ln(3)/2 = {x_star:.6f}")
print()

print("Error Tipo I (clasificar Y=0 cuando es Y=1):")
print(f"  P(X > x* | Y=1) · P(Y=1)")
print(f"  = e^(-3·x*) · 0.5")
print(f"  = e^(-3·ln(3)/2) · 0.5")
print(f"  = e^(ln(3^(-3/2))) · 0.5")
print(f"  = 3^(-3/2) · 0.5")
print(f"  = (1/√27) · 0.5")
print(f"  = {P_error_1:.6f}")
print()

print("Error Tipo II (clasificar Y=1 cuando es Y=0):")
print(f"  P(X < x* | Y=0) · P(Y=0)")
print(f"  = (1 - e^(-x*)) · 0.5")
print(f"  = (1 - e^(-ln(3)/2)) · 0.5")
print(f"  = (1 - 1/√3) · 0.5")
print(f"  = {P_error_2:.6f}")

In [ ]:
# Error total
error_total = P_error_1 + P_error_2

print("\n" + "="*60)
print("ERROR TOTAL DEL CLASIFICADOR")
print("="*60)
print(f"\nError = Error Tipo I + Error Tipo II")
print(f"Error = {P_error_1:.6f} + {P_error_2:.6f}")
print(f"Error = {error_total:.6f}")
print(f"\nError ≈ {error_total:.4f} = {error_total*100:.2f}%")
print(f"Precisión ≈ {1-error_total:.4f} = {(1-error_total)*100:.2f}%")

In [ ]:
# Forma analítica exacta del error
print("\n" + "="*60)
print("EXPRESIÓN ANALÍTICA EXACTA DEL ERROR")
print("="*60)

# x* = ln(3)/2
# Error = (1/2) * [3^(-3/2) + (1 - 3^(-1/2))]
# Error = (1/2) * [1/√27 + 1 - 1/√3]
# Error = (1/2) * [1/√27 + 1 - √3/3]

sqrt3 = np.sqrt(3)
sqrt27 = np.sqrt(27)

print(f"\nCon x* = ln(3)/2:")
print(f"")
print(f"Error = (1/2) · [3^(-3/2) + (1 - 3^(-1/2))]")
print(f"      = (1/2) · [1/√27 + 1 - 1/√3]")
print(f"      = (1/2) · [{1/sqrt27:.6f} + 1 - {1/sqrt3:.6f}]")
print(f"      = (1/2) · [{1/sqrt27 + 1 - 1/sqrt3:.6f}]")
print(f"      = {error_total:.6f}")

# Simplificación
print(f"\nSimplificando:")
print(f"Error = (1/2) · (1 + 1/√27 - 1/√3)")
print(f"      = (1/2) · (1 + 1/(3√3) - √3/3)")
print(f"      = (1/2) · (1 - (3-1)/(3√3))")
print(f"      = (1/2) · (1 - 2/(3√3))")
print(f"      = (1/2) · (1 - 2√3/9)")
print(f"      ≈ {error_total:.6f}")

In [ ]:
# Verificación numérica del error mediante integración
print("\nVERIFICACIÓN NUMÉRICA MEDIANTE INTEGRACIÓN")
print("="*60)

# Error Tipo II: integral de f0(x) desde 0 hasta x* multiplicada por P(Y=0)
error_tipo2_num, _ = integrate.quad(f0, 0, x_star)
error_tipo2_num *= P_Y0

# Error Tipo I: integral de f1(x) desde x* hasta infinito multiplicada por P(Y=1)
error_tipo1_num, _ = integrate.quad(f1, x_star, np.inf)
error_tipo1_num *= P_Y1

error_total_num = error_tipo1_num + error_tipo2_num

print(f"Error Tipo I (numérico): {error_tipo1_num:.6f}")
print(f"Error Tipo II (numérico): {error_tipo2_num:.6f}")
print(f"Error Total (numérico): {error_total_num:.6f}")
print(f"\nCoincide con el cálculo analítico: {np.isclose(error_total, error_total_num)}")

## 5. Visualización del Error

In [ ]:
# Visualización de las áreas de error
x = np.linspace(0, 4, 1000)

fig, ax = plt.subplots(figsize=(12, 7))

# Densidades ponderadas
ax.plot(x, f0(x) * P_Y0, 'b-', linewidth=2.5, label=r'$f_0(x) \cdot P(Y=0)$ (Clase 0)')
ax.plot(x, f1(x) * P_Y1, 'r-', linewidth=2.5, label=r'$f_1(x) \cdot P(Y=1)$ (Clase 1)')

# Áreas de error
# Error Tipo II: área bajo f0 desde 0 hasta x* (clasificar como 1 cuando es 0)
x_error2 = x[x <= x_star]
ax.fill_between(x_error2, 0, f0(x_error2) * P_Y0, alpha=0.4, color='blue', 
                label=f'Error Tipo II = {P_error_2:.4f}')

# Error Tipo I: área bajo f1 desde x* hasta infinito (clasificar como 0 cuando es 1)
x_error1 = x[x >= x_star]
ax.fill_between(x_error1, 0, f1(x_error1) * P_Y1, alpha=0.4, color='red',
                label=f'Error Tipo I = {P_error_1:.4f}')

# Línea de decisión
ax.axvline(x=x_star, color='green', linestyle='--', linewidth=2.5, 
           label=f'Umbral x* = {x_star:.4f}')

ax.set_xlabel('x', fontsize=14)
ax.set_ylabel('Densidad ponderada', fontsize=14)
ax.set_title(f'Clasificador Bayesiano Óptimo\nError Total = {error_total:.4f} ({error_total*100:.2f}%)', fontsize=16)
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 4)
ax.set_ylim(0, 1.6)

# Añadir anotaciones
ax.annotate('Clasificar\ncomo Y=1', xy=(x_star/2, 0.1), fontsize=12, ha='center',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax.annotate('Clasificar\ncomo Y=0', xy=((x_star+2)/2, 0.1), fontsize=12, ha='center',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('ejercicio2_error_clasificador.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Resumen y Conclusiones

In [ ]:
print("="*70)
print("RESUMEN DEL CLASIFICADOR BAYESIANO ÓPTIMO")
print("="*70)

print("\n1. DISTRIBUCIONES:")
print(f"   Clase Y=0: f₀(x) = e^(-x)     [Exponencial con λ=1]")
print(f"   Clase Y=1: f₁(x) = 3e^(-3x)   [Exponencial con λ=3]")
print(f"   P(Y=0) = P(Y=1) = 0.5")

print("\n2. REGLA DE CLASIFICACIÓN:")
print(f"")
print(f"          ⎧ 1   si x < ln(3)/2 ≈ {x_star:.4f}")
print(f"   h(x) = ⎨")
print(f"          ⎩ 0   si x ≥ ln(3)/2 ≈ {x_star:.4f}")

print("\n3. REGIONES DE CLASIFICACIÓN:")
print(f"   R₁ = [0, {x_star:.4f})  → Clasificar como Y=1")
print(f"   R₀ = [{x_star:.4f}, ∞)  → Clasificar como Y=0")

print("\n4. ERROR DEL CLASIFICADOR:")
print(f"   Error Tipo I  (Y=1 clasificado como Y=0): {P_error_1:.6f}")
print(f"   Error Tipo II (Y=0 clasificado como Y=1): {P_error_2:.6f}")
print(f"   ─────────────────────────────────────────────────")
print(f"   Error Total (Error de Bayes): {error_total:.6f}")
print(f"")
print(f"   Error ≈ {error_total*100:.2f}%")
print(f"   Precisión ≈ {(1-error_total)*100:.2f}%")

print("\n" + "="*70)

## 7. Interpretación

### Regla de Clasificación:
$$h(x) = \begin{cases} 1 & \text{si } x < \frac{\ln(3)}{2} \approx 0.5493 \\ 0 & \text{si } x \geq \frac{\ln(3)}{2} \end{cases}$$

### Regiones de Clasificación:
- **Región $R_1$**: $[0, 0.5493)$ → Clasificar como clase 1
- **Región $R_0$**: $[0.5493, \infty)$ → Clasificar como clase 0

### Error del Clasificador:
$$\text{Error de Bayes} = \frac{1}{2}\left(3^{-3/2} + 1 - 3^{-1/2}\right) \approx 0.3080 = 30.80\%$$

### Intuición:
- La distribución Exp(3) (clase 1) tiene mayor densidad cerca de cero y decae más rápidamente
- La distribución Exp(1) (clase 0) tiene menor densidad inicial pero decae más lentamente
- Por esto, valores pequeños de x se clasifican como clase 1, y valores grandes como clase 0
- El umbral $x^* = \ln(3)/2$ es donde las probabilidades ponderadas se igualan